# FinSight - 02 . Customer & Campaign Analysis

Uses the SQLite database and analytics outputs to look at RFM segments, the income->spend relationship, and campaign performance (redemption vs ROI).

Run `python src/run_pipeline.py` first so the DB and `outputs/` exist.

In [ ]:
import sqlite3
from pathlib import Path
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
sns.set_theme(style='whitegrid')
ROOT = Path.cwd().parent if Path.cwd().name=='notebooks' else Path.cwd()
conn=sqlite3.connect(ROOT/'db'/'finsight.db')
out=ROOT/'outputs'

## 1. RFM segments
Sizes and average monetary value per segment (computed from behaviour, not a pre-baked label).

In [ ]:
rfm=pd.read_sql('SELECT * FROM rfm_segments',conn)
summary=rfm.groupby('rfm_segment').agg(customers=('customer_id','count'),
  avg_recency=('recency_days','mean'),avg_freq=('frequency','mean'),avg_monetary=('monetary','mean')).round(1)
display(summary)
summary['customers'].plot(kind='bar',figsize=(8,4),title='Customers per RFM segment'); plt.tight_layout()

## 2. B1 check - income band vs average spend
Validates the documented relationship: higher income bands should spend more per transaction.

In [ ]:
q='''SELECT c.income_band, ROUND(AVG(t.transaction_amount),0) avg_amt, COUNT(*) txns
FROM customers c JOIN transactions t ON t.customer_id=c.customer_id
GROUP BY c.income_band ORDER BY avg_amt DESC'''
band=pd.read_sql(q,conn); display(band)
order=['Low','Lower-Mid','Mid','Upper-Mid','High']
b=band[band.income_band.isin(order)].set_index('income_band').reindex(order)
b['avg_amt'].plot(kind='bar',figsize=(8,4),title='Avg transaction amount by income band'); plt.tight_layout()

## 3. Campaign performance - redemption vs ROI
Each point is a campaign. Some high-redemption campaigns still have negative ROI (deep discounts).

In [ ]:
perf=pd.read_csv(out/'campaign_performance.csv')
fig,ax=plt.subplots(figsize=(9,5))
ax.scatter(perf['redemption_rate'],perf['roi'],s=perf['campaign_revenue']/2000,
   c=(perf['roi']>0),cmap='RdYlGn',edgecolor='k',alpha=0.8)
ax.axhline(0,color='grey',ls='--'); ax.set_xlabel('Redemption rate'); ax.set_ylabel('ROI')
ax.set_title('Campaigns: redemption vs ROI (size = revenue)'); plt.tight_layout()

## 4. Revenue by RFM segment

In [ ]:
q='''SELECT r.rfm_segment, ROUND(SUM(t.transaction_amount),0) revenue, COUNT(DISTINCT t.customer_id) customers
FROM transactions t JOIN rfm_segments r ON r.customer_id=t.customer_id
GROUP BY r.rfm_segment ORDER BY revenue DESC'''
seg=pd.read_sql(q,conn); display(seg)
seg.set_index('rfm_segment')['revenue'].plot(kind='barh',figsize=(8,4),title='Revenue by segment'); plt.tight_layout()

## 5. Underperforming campaigns
Lowest redemption among campaigns with a meaningful engaged base - candidates for review.

In [ ]:
weak=perf[perf['engaged']>=30].sort_values('redemption_rate').head(8)
display(weak[['campaign_id','campaign_category','engaged','redeemed','redemption_rate','roi']])
conn.close()

### Observations
- RFM cleanly separates High Value, Regular, Low Engagement and At Risk customers.
- The income->spend tendency (B1) is clearly recovered by simple SQL.
- Redemption and ROI are **not** the same story - deep-discount campaigns can be popular yet unprofitable.
- The weakest campaigns are concrete review candidates for the opportunities page.